In [ ]:
# %% [markdown]
# # Convert Index File to RDF (Task 2.3)

# %% [markdown]
# ## 1. Import required libraries

# %%
import re
from datetime import datetime
import os

# %% [markdown]
# ## 2. Define RDF prefix

# %%
def get_prefixes():##Define namespace prefix in order to tell parser the data following which ontology.(The movie ootology.ttl)
    """Return RDF prefix definition"""
    return """@prefix : <http://www.semanticweb.org/legion/ontologies/2026/0/untitled-ontology-5/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .

"""

# %% [markdown]
# ## 3. Parse index file

# %%
def parse_index_file(txt_file):
    """Parse the index file and return movie data"""
    movies = []
    
    with open(txt_file, 'r', encoding='utf-8') as f: 
        content = f.read()
    
    # Split by delimiter
    blocks = content.split('----------------------------------------')
    
    for block in blocks:
        if not block.strip() or 'ID:' not in block:
            continue
            
        movie = {}
        lines = block.strip().split('\n')
        
        for line in lines:
            line = line.strip()
            if not line:
                continue
           ##only extract data (Delete the 'character that we do not need'...eg:ID: 150540 ,we delete "ID:"——>150540)
            if line.startswith('ID:'):
                movie['id'] = line.replace('ID:', '').strip()
            elif line.startswith('Title:'):
                movie['title'] = line.replace('Title:', '').strip()
            elif line.startswith('Year:'):
                movie['year'] = line.replace('Year:', '').strip()
            elif line.startswith('Rating:'):
                movie['rating'] = line.replace('Rating:', '').strip()
            elif line.startswith('Genres:'):
                genres = line.replace('Genres:', '').strip()
                movie['genres'] = [g.strip() for g in genres.split(',')]
            elif line.startswith('GenreURIs:'):
                genre_uris = line.replace('GenreURIs:', '').strip()
                movie['genre_uris'] = [g.strip() for g in genre_uris.split(',')]
            elif line.startswith('Director:'):
                directors = line.replace('Director:', '').strip()
                movie['directors'] = [d.strip() for d in directors.split(',')]
            elif line.startswith('DirectorURIs:'):
                director_uris = line.replace('DirectorURIs:', '').strip()
                movie['director_uris'] = [d.strip() for d in director_uris.split(',')]
            elif line.startswith('Actors:'):
                actors = line.replace('Actors:', '').strip()
                movie['actors'] = [a.strip() for a in actors.split(',')]
            elif line.startswith('ActorURIs:'):
                actor_uris = line.replace('ActorURIs:', '').strip()
                movie['actor_uris'] = [a.strip() for a in actor_uris.split(',')]
            elif line.startswith('Companies:'):
                companies = line.replace('Companies:', '').strip()
                movie['companies'] = [c.strip() for c in companies.split(',')]
            elif line.startswith('CompanyURIs:'):
                company_uris = line.replace('CompanyURIs:', '').strip()
                movie['company_uris'] = [c.strip() for c in company_uris.split(',')]
            elif line.startswith('Countries:'):
                countries = line.replace('Countries:', '').strip()
                movie['countries'] = [c.strip() for c in countries.split(',')]
            elif line.startswith('CountryURIs:'):
                country_uris = line.replace('CountryURIs:', '').strip()
                movie['country_uris'] = [c.strip() for c in country_uris.split(',')]
            elif line.startswith('Plot:'):
                plot = line.replace('Plot:', '').strip()
                movie['plot'] = plot
        
        if movie.get('id'):
            movies.append(movie)
    
    print(f"Parsed {len(movies)} movies from index file")
    return movies

# %% [markdown]
# ## 4. Generate RDF/Turtle file

# %%
def generate_rdf(movies, output_file='movie_ontology.ttl'):
    """Generate RDF/Turtle file from parsed movies"""
    
    with open(output_file, 'w', encoding='utf-8') as f: ## write mode ('w')
        # Write prefixes
        f.write(get_prefixes())
        
        # Header
        f.write("#" + "="*80 + "\n")## Delimiter
        f.write(f"# Movie Ontology RDF Data\n")
        f.write(f"# Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"# Total Movies: {len(movies)}\n")
        f.write("#" + "="*80 + "\n\n")
        
        # Track added entities
        actors_added = set()
        directors_added = set()
        genres_added = set()
        companies_added = set()
        countries_added = set()##Ensure that each entity is defined only once (avoid duplication)
        
        # ========================================
        # 1. Define all Genres (with :GenreName)
        # ========================================
        f.write("#" + "-"*40 + "\n")
        f.write("# Genres\n")
        f.write("#" + "-"*40 + "\n\n")
        
        for movie in movies:
            if 'genres' in movie and 'genre_uris' in movie:
                for i, genre in enumerate(movie['genres']):
                    if i < len(movie['genre_uris']):
                        genre_uri = movie['genre_uris'][i]
                        if genre_uri and genre_uri not in genres_added:
                            f.write(f":{genre_uri} rdf:type owl:NamedIndividual , :Genre ;\n")
                            f.write(f'    :GenreName "{genre}"@en .\n\n')
                            genres_added.add(genre_uri)
        f.write("\n")
        ##Usage: The GenreName property provides a human-readable type name.The @en language tag indicates the English name.
        ##Each type is defined only once (deduplicated through the genres_added set).Referenced by movies through the :hasGenre property.
        # ========================================
        # 2. Define all Countries
        # ========================================
        f.write("#" + "-"*40 + "\n")
        f.write("# Countries\n")
        f.write("#" + "-"*40 + "\n\n")
        
        for movie in movies:
            if 'countries' in movie and 'country_uris' in movie:
                for i, country in enumerate(movie['countries']):
                    if i < len(movie['country_uris']):
                        country_uri = movie['country_uris'][i]
                        if country_uri and country_uri not in countries_added:
                            f.write(f":{country_uri} rdf:type owl:NamedIndividual , :Country .\n")
                            countries_added.add(country_uri)
        f.write("\n")
        ##Only type declaration, no data properties.The country name itself is used as part of the URI (e.g., United_States_of_America)
        ##Each country is defined only once (deduplicated through the countries_added set),Referenced by movies through the :producedIn property
        # ========================================
        # 3. Define all Companies (with :CompanyName)
        # ========================================
        f.write("#" + "-"*40 + "\n")
        f.write("# Production Companies\n")
        f.write("#" + "-"*40 + "\n\n")
        
        for movie in movies:
            if 'companies' in movie and 'company_uris' in movie:
                for i, company in enumerate(movie['companies']):
                    if i < len(movie['company_uris']):
                        company_uri = movie['company_uris'][i]
                        if company_uri and company_uri not in companies_added:
                            f.write(f":{company_uri} rdf:type owl:NamedIndividual , :Company ;\n")
                            f.write(f'    :CompanyName "{company}"@en .\n\n')
                            companies_added.add(company_uri)
        f.write("\n")
        
        # ========================================
        # 4. Define all Directors (with :name)
        # ========================================
        f.write("#" + "-"*40 + "\n")
        f.write("# Directors\n")
        f.write("#" + "-"*40 + "\n\n")
        
        for movie in movies:
            if 'directors' in movie and 'director_uris' in movie:
                for i, director in enumerate(movie['directors']):
                    if i < len(movie['director_uris']):
                        director_uri = movie['director_uris'][i]
                        if director_uri and director_uri not in directors_added:
                            f.write(f":{director_uri} rdf:type owl:NamedIndividual , :Director ;\n")
                            f.write(f'    :name "{director}"@en .\n\n')
                            directors_added.add(director_uri)
        f.write("\n")
        
        # ========================================
        # 5. Define all Actors (with :name)
        # ========================================
        f.write("#" + "-"*40 + "\n")
        f.write("# Actors\n")
        f.write("#" + "-"*40 + "\n\n")
        
        for movie in movies:
            if 'actors' in movie and 'actor_uris' in movie:
                for i, actor in enumerate(movie['actors']):
                    if i < len(movie['actor_uris']):
                        actor_uri = movie['actor_uris'][i]
                        if actor_uri and actor_uri not in actors_added:
                            f.write(f":{actor_uri} rdf:type owl:NamedIndividual , :Actor_Actress ;\n")## Align here with the subsequent SPARQL queries
                            f.write(f'    :name "{actor}"@en .\n\n')
                            actors_added.add(actor_uri)
        f.write("\n")
        
        # ========================================
        # 6. Define all Movies
        # ========================================
        f.write("#" + "-"*40 + "\n")
        f.write("# Movies\n")
        f.write("#" + "-"*40 + "\n\n")
        
        for movie in movies:
            movie_id = movie.get('id', 'unknown')
            movie_uri = f":movie_{movie_id}"
            
            f.write(f"###  Movie: {movie.get('title', 'Unknown')} (ID: {movie_id})\n")
            f.write(f"{movie_uri} rdf:type owl:NamedIndividual , :Movie ;\n")
            
            # Title
            if 'title' in movie:
                title = movie['title'].replace('"', '\\"')
                f.write(f'    :title "{title}" ;\n')
            
            # Year
            if 'year' in movie and movie['year']:
                f.write(f'    :releaseYear "{movie["year"]}"^^xsd:integer ;\n')
            
            # Rating
            if 'rating' in movie and movie['rating']:
                try:
                    rating_val = float(movie['rating'])
                    f.write(f'    :rating "{rating_val}"^^xsd:float ;\n')
                except:
                    pass
            
            # Plot
            if 'plot' in movie and movie['plot']:
                plot = movie['plot'].replace('"', '\\"')
                f.write(f'    :plot "{plot}" ;\n')
            
            # Genre relationships
            if 'genre_uris' in movie:
                for genre_uri in movie['genre_uris']:
                    if genre_uri:
                        f.write(f'    :hasGenre :{genre_uri} ;\n')
            
            # Director relationships
            if 'director_uris' in movie:
                for director_uri in movie['director_uris']:
                    if director_uri:
                        f.write(f'    :directedBy :{director_uri} ;\n')
            
            # Company relationships
            if 'company_uris' in movie:
                for company_uri in movie['company_uris']:
                    if company_uri:
                        f.write(f'    :producedBy :{company_uri} ;\n')
            
            # Country relationships
            if 'country_uris' in movie:
                for country_uri in movie['country_uris']:
                    if country_uri:
                        f.write(f'    :producedIn :{country_uri} ;\n')
            
            # Remove last ';' and end
            f.seek(f.tell() - 2, 0)
            f.write(".\n\n")
        
        # ========================================
        # 7. Actor-Film Relationships
        # ========================================
        f.write("#" + "-"*40 + "\n")
        f.write("# Actor-Film Relationships\n")
        f.write("#" + "-"*40 + "\n\n")
        
        for movie in movies:
            if 'actor_uris' in movie and 'id' in movie:
                movie_id = movie['id']
                movie_uri = f":movie_{movie_id}"
                
                for actor_uri in movie['actor_uris']:
                    if actor_uri:
                        f.write(f":{actor_uri} :actedIn {movie_uri} .\n")
        f.write("\n")
        
        # ========================================
        # 8. Director-Film Relationships
        # ========================================
        f.write("#" + "-"*40 + "\n")
        f.write("# Director-Film Relationships\n")
        f.write("#" + "-"*40 + "\n\n")
        
        for movie in movies:
            if 'director_uris' in movie and 'id' in movie:## check id in movie or not
                movie_id = movie['id']## get movie id 
                movie_uri = f":movie_{movie_id}"## create movie uri ...eg:movie_4133
                
                for director_uri in movie['director_uris']:##Traverse all of this movie
                    if director_uri:## vertify director_uri exist
                        f.write(f":{director_uri} :directed {movie_uri} .\n")## Write in triple 
    
    print(f"RDF file saved as: {output_file}")
    return output_file

# %% [markdown]
# ## 5. Running reminding

# %%

output_filename = 'movie_ontology_rdf.ttl'

# Parse index file
print("\nParsing index file...")
movies = parse_index_file('movie_index.txt')

# Generate RDF
print(f"\n Generating RDF for {len(movies)} movies...")
rdf_file = generate_rdf(movies, output_filename)

print(f"\nRDF file generated: {rdf_file}")


Parsing index file...
Parsed 200 movies from index file

 Generating RDF for 200 movies...
RDF file saved as: movie_ontology_rdf.ttl

RDF file generated: movie_ontology_rdf.ttl
